In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from daemon_analysis_tools.io.csv_handler import load_and_process_csv
from daemon_analysis_tools.io.yaml_handler import save_answers_to_yaml, load_answers_from_yaml
from daemon_analysis_tools.processing.grouper import group_questions_by_journal
from daemon_analysis_tools.services.discrepancy_resolver import resolve_discrepancy

Load and process data:
- Group answers by publisher and journal, trying to uniform names written in slightly different ways.
- Store in a DataFrame

In [3]:
data = load_and_process_csv("../../data/raw/rdp.csv")

Get a `dict` labeled by publisher names of `dict`s labeled by journal names of `dict`s of `Question` instances. The `.answer` attribute contains the answers given by the respondents and the explanations text to motivate it.

In [4]:
question_metadata_file = "../../data/metadata/question_metadata.yaml"

grouped_questions = group_questions_by_journal(data, question_metadata_file)

## Resolve discrepancies

The `Question` class has a `.resolve_discrepancies` method which updates `Question.anwsers` with the correct answer.

For example, let's consider IOP's 2D Materials. Question 7 has discrepancies.

In [5]:
for journal, data in grouped_questions["Elsevier"].items():
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies():
            answer.print_qa()
    print("\n\n")

applied_catalysis_b_environment_and_energy
3. Data sharing requirements in RDP
  Resp. 0:
    Answer: Data sharing encouraged but optional.
    Explanation: For this journal, the following instructions from our research data guidelines apply.
Option C: Research data deposit, citation and linking
You are required to:
• Deposit your research data in a relevant data repository.
• Cite and link to this dataset in your article.
• If this is not possible, make a statement explaining why research data cannot be shared.

p.s. data statements such as: „The data that has been used is confidential.” are found in articles. (Only options D and E from research data guidelines are actually requiring data, and option C which is relevant here states require but offers alternative)
  Resp. 1:
    Answer: Data sharing required but not publicly (e.g. available upon request is allowed).
    Explanation: You are required to:

Deposit your research data in a relevant data repository.

Cite and link to this d

Inconsistencies can be removed manually, passing the index of the correct respondent.

In [6]:
for j in ["chemical_engineering_journal", ]:
    for i in [2, 3]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=0,
            discrepancy_reason="Text not found",
        )

    for i in [8, 14, 15]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )
    
    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=1,
            discrepancy_reason="Formatting",
        )

for j in ["composites_science_and_technology", "international_journal_of_thermal_sciences", "optical_materials",
          "applied_catalysis_b_environment_and_energy"]:
    for i in [3]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=0,
            discrepancy_reason="Text not found",
        )

    for i in [8, 14, 15]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )
    
    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=1,
            discrepancy_reason="Formatting",
        )

for j in ["energy_storage_materials", ]:
    for i in [2]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=0,
            discrepancy_reason="Text not found",
        )

    for i in [10]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=0,
            discrepancy_reason="Language understanding",
        )

    for i in [8, 14, 15]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=1,
            discrepancy_reason="Language understanding",
        )
    
    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=1,
            discrepancy_reason="Formatting",
        )

for j in ["escience", ]:
    for i in [1, 2, 4, 8, 10]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=1,
            discrepancy_reason="Text not found",
        )
    
    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["Elsevier"][j][i],
            correct_answer=1,
            discrepancy_reason="Formatting",
        )

In [7]:
for journal, data in grouped_questions["Elsevier"].items():
    print("#############################################################")
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies() and answer.correct_answer is None:
            answer.print_qa()

#############################################################
applied_catalysis_b_environment_and_energy
#############################################################
chemical_engineering_journal
#############################################################
composites_science_and_technology
#############################################################
energy_storage_materials
#############################################################
escience
#############################################################
international_journal_of_thermal_sciences
#############################################################
materials_today_proceedings
#############################################################
optical_materials
#############################################################
ssrn_electronic_journal


In [8]:
save_answers_to_yaml(
    grouped_questions,
    parent_folder="../../data/processed/all_answers",
    save_only=["Elsevier"],
)

../../data/processed/all_answers/Elsevier/applied_catalysis_b_environment_and_energy.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete this file if necessary.
../../data/processed/all_answers/Elsevier/chemical_engineering_journal.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete this file if necessary.
../../data/processed/all_answers/Elsevier/composites_science_and_technology.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete this file if necessary.
../../data/processed/all_answers/Elsevier/energy_storage_materials.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete this file if necessary.
../../data/processed/all_answers/Elsevier/escience.yaml already exists. No data was written to prevent overwriting files modified by users. Manually delete this file if necessary.
../..

After doing this, the `.get_final_answer()` method returns the correct answer.